# Diabetes Dataset

In [19]:
import pandas as pd
import numpy as np

In [20]:
df = pd.read_csv("diabetes.csv")

#### Q1. What is the objective of the diabetes dataset?

The objective is to predict whether a patient has diabetes or not based on diagnostic and physiological measurements.

#### Q2. How many instances and attributes are present?

In [21]:
print("Instances: ", df.shape[0])
print("Attributes: ", df.shape[1])

Instances:  768
Attributes:  9


#### Q3. Which attributes are numerical and which are categorical?

Numerical Attributes: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age
Categorical Attributes: Outcome

#### Q4. What does the target variable represent?

The target variable 'Outcome' represents whether the patient is diagnosed with diabetes or not. 
- If the outcome is '0', patient does not have diabetes.
- If the outcome is '1', patient has diabetes.

#### Q5. Are there any missing or zero-valued attributes? How should they be treated?

Yes, the dataset contains zero-valued attributes. Some zeros, such as `Pregnancies = 0` and `Outcome = 0`, are valid values. 
However, zeros in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` generally represent missing or unavailable measurements. These should be treated as missing values and can be replaced using appropriate imputation techniques, such as the median or by 'NaN'. Median is preferable to mean here because medical variables such as insulin can contain substantial outliers.

In [22]:
missingCols = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

for col in missingCols:
    df[col] = df[col].fillna(df[col].median())

#### Q6. What is the class distribution? (diabetic vs non-diabetic)?

In [23]:
print(df["Outcome"].value_counts())

print("\nPercentage:")
print(df["Outcome"].value_counts(normalize=True) * 100)

Outcome
0    500
1    268
Name: count, dtype: int64

Percentage:
Outcome
0    65.104167
1    34.895833
Name: proportion, dtype: float64


#### Q7. Is the dataset balanced or unbalanced?

The dataset is imbalanced. A balanced binary dataset would have approximately: 50% class 0 and 50% class 1. But, here we have 65.10% for class 0 and 34.89% for class 1.

#### Q8. How do you handle missing values in glucose, insulin, BMI, or blood pressure?

In [24]:
# We first convert the invalid zeroes to missing values
missingCols = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

df[missingCols] = df[missingCols].replace(0, np.nan)


# We then check for the number of missing values
print(df.isnull().sum())


# Impute the median for the missing values
for col in missingCols:
    df[col] = df[col].fillna(df[col].median())

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


#### Q9. Why should zero values in medical attributes be treated as missing data?

Values for Glucose, BMI, BloodPressure are generally physiologically implausible for a living patient. Therefore, they most likely mean that the measurement was not recorded.
If we treat them as normal values, they can severely distort the mean, variance of the data.

#### Q10. What normalization or standardization techniques are suitable?

The most suitable technique here is usually standardization. This helps transform each feature so that approximately mean = 0 and standard deviation = 1.

#### Q11. How do you handle outliers in medical datasets?

Outliers in medical datasets have to first be detected. We can detect them using the IQR method which is the interquartile range method

In [25]:
numeric_features = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age"
]

for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(col, ":", len(outliers), "outliers")

Pregnancies : 4 outliers
Glucose : 0 outliers
BloodPressure : 14 outliers
SkinThickness : 87 outliers
Insulin : 346 outliers
BMI : 8 outliers
DiabetesPedigreeFunction : 29 outliers
Age : 9 outliers


#### Q12. Is feature scaling required for all classifiers? Why?

No. Feature scaling is important for models whose calculations depend on feature magnitude or distance.

#### Q13. How do you encode categorical features, if present?

There are no categorical input features in the standard dataset. Usually, for a categorical feature, one-hot encoding is preferred.
The existing 'Outcome' variable does not need one-hot encoding for ordinary binary classification. It can remain 0 / 1.

#### Q14. What is the mean and variance of glucose levels for diabetic vs non-diabetic patients?

In [26]:
group_stats = df.groupby("Outcome")["Glucose"].agg(
    ["count", "mean", "var", "std"]
)

print(group_stats)

         count        mean         var        std
Outcome                                          
0          500  110.682000  610.445768  24.707201
1          268  142.130597  874.316214  29.568839


#### Q15. Is there a significant correlation between BMI and diabetes?

In [27]:
correlation = df["BMI"].corr(df["Outcome"])

print("BMI-Outcome correlation:", correlation)

BMI-Outcome correlation: 0.3120383386771113


#### Q16. Which features show the strongest correlation with diabetes?

In [28]:
correlation = df.corr(numeric_only=True)["Outcome"].sort_values(
    ascending=False
)

print(correlation)

Outcome                     1.000000
Glucose                     0.492782
BMI                         0.312038
Age                         0.238356
Pregnancies                 0.221898
SkinThickness               0.214873
Insulin                     0.203790
DiabetesPedigreeFunction    0.173844
BloodPressure               0.165723
Name: Outcome, dtype: float64


#### Q17. Perform a hypothesis test to check whether glucose level significantly affects diabetes outcome.

We compare the mean glucose levels of patients with `Outcome = 0` and `Outcome = 1` using an independent two-sample t-test. The null hypothesis assumes that both groups have the same mean glucose level. Since the calculated p-value is less than 0.05, we reject the null hypothesis and conclude that glucose level has a statistically significant relationship with diabetes outcome.

#### Q18. What statistical test would you use to compare two groups of patients?

If we want to compare a continuous variable such as Glucose, BMI, or BloodPressure between diabetic (Outcome = 1) and non-diabetic (Outcome = 0) patients, we can use an independent two-sample t-test. If the variances are unequal, Welch's t-test is preferred. For non-normal data, the Mann-Whitney U test can be used.

#### Q19. Which features contribute most to diabetes prediction?

Glucose contributes the most to predicting diabetes, followed by features such as BMI, Age, Pregnancies, and DiabetesPedigreeFunction. Glucose has the strongest correlation with Outcome, indicating that patients with higher glucose levels are more likely to have a positive diabetes outcome.

#### Q20. How does feature selection improve model performance?

Feature selection can remove less useful attributes such as SkinThickness or BloodPressure when they provide little predictive information, while retaining stronger features such as Glucose, BMI, Age, and Pregnancies. This reduces unnecessary data, noise, and the risk of overfitting, and can make the diabetes prediction model simpler and easier to interpret.

#### Q22. Can PCA be applied to this dataset? Why or why not?

Yes, PCA (Principal Component Analysis) can be applied to the Diabetes dataset because its eight input features are numerical. PCA can reduce the dimensionality by converting features such as Glucose, BMI, Age, Insulin, and BloodPressure into a smaller number of principal components while retaining most of the variance. The features should be standardized before applying PCA because they have different scales.

#### Q23. How do domain-specific features improve prediction accuracy?

Domain-specific feature engineering can create more meaningful features from existing medical measurements. For example, BMI categories, glucose-level categories, age groups, or an interaction such as Glucose × BMI can help the model capture medically relevant patterns.

In [29]:
# Example BMI Category

df["BMI_Category"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 25, 30, np.inf],
    labels=[
        "Underweight",
        "Normal",
        "Overweight",
        "Obese"
    ]
)